# Phase 2 RAPID Smoke Tests

This notebook is for small, local checks of the Phase 2 RAPID extraction.

It covers:
- the shared single-edge control artifact
- the extracted `rapid_tools` adapter layer
- RAPID prep-file generation through the shared `rapid_tools.prep` module
- a tiny end-to-end RAPID run on the explicit single-edge control
- prep-file generation on a tiny adapted synthetic loop network

Use a kernel/environment that has the RAPID dependencies installed.
The `UNC` environment is the intended one for these checks.


In [1]:
from pathlib import Path
import sys
import shutil
import json
import pandas as pd
from IPython.display import display


def find_repo_root() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/Users/6256481/Code/river-hierarchy"),
    ]
    for candidate in candidates:
        if (candidate / "synthetic_runs" / "src" / "synthetic_runs").exists() and (candidate / "RAPID" / "src" / "rapid_tools").exists():
            return candidate.resolve()
    raise RuntimeError("Could not locate repo root.")


ROOT = find_repo_root()
SYN_ROOT = ROOT / "synthetic_runs"
RAPID_ROOT = ROOT / "RAPID"
SYN_SRC = SYN_ROOT / "src"
RAPID_SRC = RAPID_ROOT / "src"
for path in [SYN_SRC, RAPID_SRC]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

OUT = SYN_ROOT / "outputs" / "test_phase_2"
OUT.mkdir(parents=True, exist_ok=True)
CONTROL_PATH = SYN_ROOT / "configs" / "single_edge_control.json"

print("ROOT:", ROOT)
print("OUT:", OUT)
print("CONTROL_PATH exists:", CONTROL_PATH.exists())


ROOT: /Users/6256481/Code/river-hierarchy
OUT: /Users/6256481/Code/river-hierarchy/synthetic_runs/outputs/test_phase_2
CONTROL_PATH exists: True


In [2]:
from synthetic_runs.core import Params, RiverNetworkNX
from synthetic_runs.runners import load_single_edge_control, resolve_single_edge_control
from rapid_tools.adapters.synthetic import build_single_edge_graph, rivernetwork_to_rapid_graph
from rapid_tools.prep import (
    create_conn_file,
    create_riv_file,
    compute_reach_ratios,
    compute_area_csv,
    create_routing_parameters,
    compute_dt_from_K,
    create_runoff,
)
from rapid_tools.engine import run_rapid


## 1. Explicit Single-Edge Control Artifact

This checks that the shared baseline control can be loaded and resolved
against experiment metadata.


In [3]:
single_edge_spec, single_edge_control_path = load_single_edge_control(CONTROL_PATH)
resolved_single_edge = resolve_single_edge_control(
    single_edge_spec,
    {"L": 12000.0, "W_total": 650.0},
)

print("control path:", single_edge_control_path)
resolved_single_edge


control path: /Users/6256481/Code/river-hierarchy/synthetic_runs/configs/single_edge_control.json


{'control_type': 'single_edge',
 'network_id': -1,
 'geometry_id': -1,
 'sample_type': 'single_edge',
 'network_type': 'single_edge',
 'length_m': 12000.0,
 'width_m': 650.0}

## 2. Tiny Synthetic-To-RAPID Adapter Check

This converts a very small synthetic loop network into a RAPID-ready graph.


In [4]:
tiny_p = Params(
    L=3000,
    W_total=40.0,
    xs=500,
    xe=2500,
    jump=500,
    max_breaks=1,
    min_width=10.0,
    width_step=10,
    x_stability=0.3,
)

loop_net = RiverNetworkNX(tiny_p)
loop_net.instantiate_corridor(20.0, 20.0)
loop_net.add_loop(branch="A", xb=1000.0, xr=1500.0, W1=10.0, W2=10.0, replace_corridor=True)

G_loop = rivernetwork_to_rapid_graph(loop_net)
print("adapter nodes/edges:", len(G_loop.nodes), len(G_loop.edges))

edge_rows = []
for u, v, k, data in G_loop.edges(keys=True, data=True):
    edge_rows.append(
        {
            "reach_id": data["reach_id"],
            "kind": data.get("kind"),
            "branch": data.get("branch"),
            "from_branch": data.get("from_branch"),
            "to_branch": data.get("to_branch"),
            "width": data.get("width"),
            "length": data.get("length"),
        }
    )

pd.DataFrame(edge_rows).sort_values("reach_id").reset_index(drop=True)


adapter nodes/edges: 6 7


,reach_id,kind,branch,from_branch,to_branch,width,length
0,1,main,main,,,40.0,500.0
1,2,corridor,A,,,20.0,500.0
2,3,corridor,B,,,20.0,2000.0
3,4,loop,,A,A,10.0,500.0
4,5,loop,,A,A,10.0,500.0
5,6,corridor,A,,,20.0,1000.0
6,7,main,main,,,40.0,500.0


## 3. Shared RAPID Prep On The Explicit Single-Edge Control

This creates the full RAPID prep-file bundle using only the shared Phase 2
modules.


In [5]:
single_edge_dir = OUT / "single_edge_control"
if single_edge_dir.exists():
    shutil.rmtree(single_edge_dir)
single_edge_dir.mkdir(parents=True, exist_ok=True)

G_single = build_single_edge_graph(
    resolved_single_edge["length_m"],
    resolved_single_edge["width_m"],
)
create_conn_file(G_single, directory=str(single_edge_dir) + "/")
single_riv_order = create_riv_file(G_single, directory=str(single_edge_dir) + "/")
compute_reach_ratios(G_single, use_widths=True, directory=str(single_edge_dir) + "/")
compute_area_csv(G_single, directory=str(single_edge_dir) + "/")
single_k_by_rid = create_routing_parameters(G_single, directory=str(single_edge_dir) + "/", xfc_value=0.1)
single_dt = compute_dt_from_K(str(single_edge_dir) + "/", 0.1, True)
single_times = create_runoff(
    G_single,
    directory=str(single_edge_dir) + "/",
    dt_seconds=int(single_dt),
    runOffC=[1e-10, 2e-10, 1e-10],
    return_times=True,
)

print("single-edge dt:", int(single_dt))
print("single-edge riv order:", single_riv_order)
print("generated files:", sorted(p.name for p in single_edge_dir.iterdir()))
print("k_by_rid:", single_k_by_rid)

pd.read_csv(single_edge_dir / "conn.csv", header=None)


single-edge dt: 7820
single-edge riv order: [1]
generated files: ['conn.csv', 'coords.csv', 'inflow.nc', 'kfc.csv', 'rapid_catchment.csv', 'rat.csv', 'rat_srt.csv', 'riv.csv', 'xfc.csv']
k_by_rid: {1: 7824.89106395963}


,0,1,2
0,1,0,0


## 4. Tiny End-To-End RAPID Run On The Single-Edge Control

This is the smallest full Phase 2 routing smoke test.


In [6]:
single_qout_path, single_qout = run_rapid(
    str(single_edge_dir),
    ROUTING_TIMESTEP_SECONDS=int(single_dt),
    runType="phase2",
    seed=1,
    return_qout=True,
)

print("qout path:", single_qout_path)
print("qout shape:", single_qout.shape)

pd.DataFrame(
    {
        "time": single_times,
        "Q": single_qout[:, 0],
    }
)


qout path: /Users/6256481/Code/river-hierarchy/synthetic_runs/outputs/test_phase_2/single_edge_control/Qout_MS_b82_20150101_20240531_GLDASv21_ens_dtR10800phase2_1.nc
qout shape: (3, 1)


,time,Q
0,0,0.000000
1,7820,592.050659
2,15640,1353.428711


## 5. Shared RAPID Prep On A Tiny Adapted Loop Network

This checks that the extracted adapter and prep layers also work together on
a small non-trivial synthetic network.


In [7]:
loop_dir = OUT / "tiny_loop_prep"
if loop_dir.exists():
    shutil.rmtree(loop_dir)
loop_dir.mkdir(parents=True, exist_ok=True)

create_conn_file(G_loop, directory=str(loop_dir) + "/")
loop_riv_order = create_riv_file(G_loop, directory=str(loop_dir) + "/")
compute_reach_ratios(G_loop, use_widths=True, directory=str(loop_dir) + "/")
compute_area_csv(G_loop, directory=str(loop_dir) + "/")
loop_k_by_rid = create_routing_parameters(G_loop, directory=str(loop_dir) + "/", xfc_value=0.1)
loop_dt = compute_dt_from_K(str(loop_dir) + "/", 0.1, True)

print("loop-network dt:", int(loop_dt))
print("loop-network riv order:", loop_riv_order)
print("generated files:", sorted(p.name for p in loop_dir.iterdir()))
print("k_by_rid keys:", sorted(loop_k_by_rid))

pd.read_csv(loop_dir / "riv.csv", header=None)


loop-network dt: 3210
loop-network riv order: [1, 2, 3, 4, 5, 6, 7]
generated files: ['conn.csv', 'coords.csv', 'kfc.csv', 'rapid_catchment.csv', 'rat.csv', 'rat_srt.csv', 'riv.csv', 'xfc.csv']
k_by_rid keys: [1, 2, 3, 4, 5, 6, 7]


,0
0,1
1,2
2,3
3,4
4,5
5,6
6,7
